# Реакции пользователя

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit # для разделения с учетом групп и пропорций классов
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from solution_utils import(read_dialogs, read_labels, get_groups, clean_normalize_dialog, calculate_metrics, SEED)

In [2]:
DATA_PATH = "hw1_train.jsonl"
LABELS = np.array(["positive", "negative", "neutral"])

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))
reactions = read_labels(DATA_PATH, "reaction")
labeled_dialogs = [d for d in dialogs if reactions.get(d.get('dialog_id')) is not None]
print("размеченных диалогов", len(labeled_dialogs))

диалогов: 10000
размеченных диалогов 3000


In [3]:
user_texts = [clean_normalize_dialog(d) for d in labeled_dialogs]
n_gr, groups_ids = get_groups(user_texts)
groups_ids = np.array(groups_ids)
print(f"групп: {n_gr}")
sizes = np.bincount(groups_ids)
print(f"групп >1: {(sizes > 1).sum()}")
print(f"макс размер: {sizes.max()}")
print(f"диалогов в группах >1: {sizes[sizes > 1].sum()}")

групп: 2978
групп >1: 17
макс размер: 4
диалогов в группах >1: 39


Разделяем на train val и test с учетом баланса классов и группировки

In [4]:
X = np.array(user_texts)
y = np.array([reactions[d["dialog_id"]] for d in labeled_dialogs])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss.split(X, y, groups_ids))
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=SEED)
idx_train, idx_val = next(gss.split(X[idx_train_val], y[idx_train_val], groups_ids[idx_train_val]))

X_train, y_train = X[idx_train_val[idx_train]], y[idx_train_val[idx_train]] # 60%
X_val, y_val = X[idx_train_val[idx_val]], y[idx_train_val[idx_val]] # 20%
X_test, y_test = X[idx_test], y[idx_test] # 20%

Получаем Tf-idf для всех данных

In [5]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # добавляем биграммы
    sublinear_tf=True,     # уменьшает скорость роста влияния слова от его частоты
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

## Базовые модели

### Наивный Байес

Мы на семинаре использовали MultinomialNB, но сейчас больше подойдет ComplementNB, так как он специально спроектирован для несбалансированных данных, как у нас

In [6]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 0.9, 1.0, 1.5]
metrics_NB = []
for alpha in alphas:
    nb = ComplementNB(alpha=alpha)
    nb.fit(X_train_tfidf, y_train)
    y_pred = nb.predict(X_val_tfidf)
    metrics_NB.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_NB_df = pd.DataFrame(metrics_NB, index=alphas)
metrics_NB_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.621035,0.437770,0.428598,0.588169,"[[2, 18, 48], [7, 85, 89], [10, 55, 285]]","{'precision_positive': 0.10526315789473684, 'r..."
0.005,0.627713,0.449086,0.441967,0.597119,"[[3, 19, 46], [7, 89, 85], [9, 57, 284]]","{'precision_positive': 0.15789473684210525, 'r..."
0.010,0.631052,0.450991,0.443557,0.598748,"[[3, 19, 46], [6, 89, 86], [7, 57, 286]]","{'precision_positive': 0.1875, 'recall_positiv..."
0.020,0.634391,0.447168,0.436982,0.597214,"[[2, 17, 49], [4, 87, 90], [6, 53, 291]]","{'precision_positive': 0.16666666666666666, 'r..."
0.050,0.637730,0.438009,0.425232,0.591940,"[[1, 12, 55], [3, 79, 99], [4, 44, 302]]","{'precision_positive': 0.125, 'recall_positive..."
0.100,0.647746,0.431771,0.413350,0.588834,"[[0, 8, 60], [1, 70, 110], [0, 32, 318]]","{'precision_positive': 0.0, 'recall_positive':..."
0.200,0.647746,0.424657,0.405263,0.581773,"[[0, 5, 63], [0, 62, 119], [0, 24, 326]]","{'precision_positive': 0.0, 'recall_positive':..."
0.500,0.647746,0.418432,0.397505,0.575089,"[[0, 2, 66], [0, 55, 126], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
0.900,0.619366,0.387124,0.355322,0.532757,"[[0, 2, 66], [0, 38, 143], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
1.000,0.622705,0.389029,0.357158,0.535213,"[[0, 2, 66], [0, 38, 143], [0, 15, 335]]","{'precision_positive': 0.0, 'recall_positive':..."


По всем параметрам лучше всего alpha = 0.01

In [7]:
alpha_NB = 0.01
nb = ComplementNB(alpha=alpha_NB).fit(X_train_tfidf, y_train)